# Parse Pschyrembel Dictonary

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from pathlib import Path

root_dir = Path.home() / 'scratch/projects/befunddolmetscher'

In [ ]:
import sys

sys.path.append((root_dir / 'search').as_posix())

In [ ]:
from tqdm.auto import tqdm
import json
import re
import unicodedata

def normalize(s):
    return unicodedata.normalize('NFKD', s).strip()

records = []
for path in tqdm((root_dir / 'data/Pschyrembel').glob('*.json')):
    with path.open('r') as f:
        d = json.load(f)
        url = d['url'].rstrip('/')
        if url.endswith('doc'):
            url = url[:-4]
        article_id = Path(url).stem
        assert len(article_id) == 5, article_id
        kwargs = {}
        for r in d["lex_info"][1:]:
            if ':' in r:
                k, v = r.split(':', 1)
                kwargs[k.strip()] = normalize(v)
            else:
                kwargs["lex_info"] = r.strip()
        assert 0 < len(d["breadcrumbs"]), len(d["breadcrumbs"])
        title = normalize(d["title"])
        record = {
            "id": article_id,
            "url": url,
            "date_crawled": d["time_crawled"].split('T')[0],
            "title": title,
            'title_clean': re.sub(r'\s\[[^\]]+\]$', '', title),
            "text": normalize(d["text"]),
            "genus": d["genus"],
            "numerus": d["numerus"],
            "section": d["breadcrumbs"][0],
            "chapter": d["breadcrumbs"][1] if len(d["breadcrumbs"]) > 1 else None,
            "topic": d["breadcrumbs"][2] if len(d["breadcrumbs"]) > 2 else None,
            "lex_info": d["lex_info"][0] if len(d["lex_info"]) > 0 else None,
        }
        record.update(kwargs)
        if 'ICD 10' in record and len(record['ICD 10']) > 0:
            icd = []
            match = re.finditer(r'([A-Z]\d{2}(\.\d{1,4})?!?)\W', record['ICD 10'])
            if match:
                for m in match:
                    icd.append(m.group(1))
            record['ICD 10'] = ', '.join(icd)
        records.append(record)
len(records)

flatten synonyms

In [ ]:
import pandas as pd
from copy import deepcopy

data_map = {record['id']: record for record in records }
df = pd.DataFrame.from_records(records)
df = df.drop_duplicates(subset=['id', 'title_clean'])
new_records_de = []
for name, row in df.iterrows():
    if isinstance(row['Synonym'], str) and len(row['Synonym']) > 0:
        for s in row['Synonym'].split(','):
            if s.strip() != row['title_clean']:
                record = deepcopy(data_map[row['id']])
                record['title'] = s.strip()
                record['title_clean'] = s.strip()
                new_records_de.append(record)
df_de = pd.DataFrame.from_records(records + new_records_de)
df_de = df_de.drop_duplicates(subset=['id', 'title_clean']) # keep first
df_de.to_csv(root_dir / 'data/pschyrembel_de.tsv', sep="\t", index=False)

flatten translations

In [ ]:
new_records_en = []
for name, row in df.iterrows():
    if isinstance(row['Englisch'], str) and len(row['Englisch']) > 0 and ', ' in row['Englisch']:
        for s in row['Englisch'].split(','):
            record = deepcopy(data_map[row['id']])
            record['Englisch'] = s.strip()
            new_records_en.append(record)

df_en = pd.DataFrame.from_records(records + new_records_en)
df_en = df_en[~df_en.Englisch.isna()]
df_en = df_en[~df_en.Englisch.str.contains(', ')]
df_en = df_en.drop_duplicates(subset=['id', 'Englisch']) # keep first
df_en.to_csv(root_dir / 'data/pschyrembel_en.tsv', sep="\t", index=False)

In [ ]:
import pandas as pd
dfs = {
    'de': pd.read_csv(root_dir / 'data/pschyrembel_de.tsv', sep="\t", index_col=False).fillna(''),
    'en': pd.read_csv(root_dir / 'data/pschyrembel_en.tsv', sep="\t", index_col=False).fillna(''),
}
dfs['de'].head(5)

In [ ]:
import requests
def embed(contents: list, host: str="localhost", port: str="8080"):
    """embed texts"""
    response = requests.post(
        f"http://{host}:{port}/embedding",
        headers={"Content-Type": "application/json"},
        json={"content": contents},
    )
    return response.json()

In [ ]:
from tqdm.auto import tqdm
import numpy as np
import json

def embed_pschyrembel(df, lang):
    output_path = (root_dir / f'results/pschyrembel_embed_{lang}.jsonl')
    key = 'title_clean' if lang == 'de' else 'Englisch'
    batch_size = 32
    for _, group in tqdm(df.groupby(np.arange(len(df))//batch_size), total=(len(df)//batch_size +1)):
        ids = group['id'].to_list()
        titles = group[key].to_list()
        texts = []
        for name, row in group.iterrows():
            texts.append((row[key] + '\n\n' + row['text']).strip())
        
        assert len(texts) == len(ids) == len(titles)
    
        vectors = embed(contents=texts, host='localhost', port='8181')
        assert len(vectors) == len(texts)
    
        # save in json file
        for v, i, t in zip(vectors, ids, titles):
            with output_path.open('a') as f:
                v['id'] = i
                v['title'] = t
                v['language'] = lang
                f.write(json.dumps(v) + '\n')

In [ ]:
for lang in ['de', 'en']:
    embed_pschyrembel(dfs[lang], lang)

In [ ]:
import json

vectors = []
for lang in ['de', 'en']:
    output_path = (root_dir / f'results/pschyrembel_embed_{lang}.jsonl')
    vectors.extend([json.loads(l) for l in output_path.read_text().splitlines()])

vectors = {(v['id'], v['title'], v['language']): v['embedding'][0] for v in vectors if len(v['embedding'][0]) == 4096}
len(vectors)

In [ ]:
from tqdm.auto import tqdm

titles = {'en': [], 'de': []}
texts = {'en': [], 'de': []}
metadatas = {'en': [], 'de': []}
embeds = {'en': [], 'de': []}

for lang in ['de', 'en']:
    key = 'title_clean' if lang == 'de' else 'Englisch'
    for name, row in tqdm(dfs[lang].iterrows()):
        titles[lang].append(row[key])
        texts[lang].append(row['text'])
        metadata = {'pschyrembel_id': row['id'], 'icd10': row['ICD 10'].lower()}
        metadata['german' if lang == 'en' else 'english'] = row[key]
        metadatas[lang].append(metadata)
        embeds[lang].append(vectors[(row['id'], row[key], lang)])

len(titles['de']), len(titles['en'])

In [ ]:
from sqlitevec import SQLiteVec

pschyrembel_table = {}
for lang in ['en', 'de']:
    pschyrembel_table[lang] = SQLiteVec.from_texts(
        titles=titles[lang],
        texts=texts[lang],
        embeds=embeds[lang],
        metadatas=metadatas[lang],
        table=f"pschyrembel_{lang}",
        db_file="pschyrembel.db",
        lang=lang,
    )

In [ ]:
from IPython.display import display, HTML

found = pschyrembel_table['en'].fulltext_search_with_highlight(
    query='(title: Appendicitis) OR (text: Appendicitis)' #lemma match
)
for f in found: # relevance rank from BM25
    print(f['rank'], f['pschyrembel_id'], f['icd10'], f['german'])
    display(HTML(f['title']))
    display(HTML(f['text']))
    print('---')

In [ ]:
found = pschyrembel_table['de'].fulltext_search_with_highlight(
    query='(title: Appendizitis) OR (text: Appendizitis)' #lemma match
)
for f in found: # relevance rank from BM25
    print(f['rank'], f['pschyrembel_id'], f['icd10'], f['english'])
    display(HTML(f['title']))
    display(HTML(f['text']))
    print('---')

In [ ]:
found = pschyrembel_table['de'].fulltext_search_with_highlight(
    query='(title: "Blinddarmentzündung") OR (text: "Blinddarmentzündung")' #lemma match
)
for f in found: # relevance rank from BM25
    print(f['rank'], f['pschyrembel_id'], f['icd10'], f['english'])
    display(HTML(f['title']))
    display(HTML(f['text']))
    print('---')